In [1]:
import pandas as pd
import numpy as np

In [2]:
from google.colab import files
uploaded = files.upload()

Saving features_dataset.csv to features_dataset.csv


In [3]:
df = pd.read_csv("features_dataset.csv")
df.head()

,date,team,opponent,team_goals,opponent_goals,tournament,city,country,neutral,result,...,win,team_avg_goals_last5,team_avg_conceded_last5,team_win_rate_last5,opp_avg_goals_last5,opp_avg_conceded_last5,opp_win_rate_last5,goals_diff_last5,conceded_diff_last5,win_rate_diff
0,1877-03-03,England,Scotland,1.0,3.0,Friendly,London,England,False,Away Win,...,0,1.4,1.8,0.2,2.6,1.4,0.6,-1.2,0.4,-0.4
1,1877-03-03,Scotland,England,3.0,1.0,Friendly,London,England,False,Away Win,...,1,2.6,1.4,0.6,1.4,1.8,0.2,1.2,-0.4,0.4
2,1878-03-02,Scotland,England,7.0,2.0,Friendly,Glasgow,Scotland,False,Home Win,...,1,2.8,0.6,0.8,1.6,2.4,0.2,1.2,-1.8,0.6
3,1878-03-02,England,Scotland,2.0,7.0,Friendly,Glasgow,Scotland,False,Home Win,...,0,1.6,2.4,0.2,2.8,0.6,0.8,-1.2,1.8,-0.6
4,1879-04-05,England,Scotland,5.0,4.0,Friendly,London,England,False,Home Win,...,1,1.4,3.2,0.2,5.0,0.6,1.0,-3.6,2.6,-0.8


In [4]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

features = [
    'team_avg_goals_last5',
    'team_avg_conceded_last5',
    'opp_avg_goals_last5',
    'opp_avg_conceded_last5',
    'goals_diff_last5',
    'conceded_diff_last5',
    'win_rate_diff'
]

X = df[features]
y = df['win']

model = RandomForestClassifier(random_state=42)
model.fit(X, y)

RandomForestClassifier(random_state=42)

In [5]:
def predict_match(team, opponent):
    team_row = latest_team_stats[latest_team_stats['team'] == team]
    opp_row = latest_team_stats[latest_team_stats['team'] == opponent]

    if team_row.empty or opp_row.empty:
        return "Team name not found. Check spelling."

    team_row = team_row.iloc[0]
    opp_row = opp_row.iloc[0]

    match_features = pd.DataFrame([{
        'team_avg_goals_last5': team_row['team_avg_goals_last5'],
        'team_avg_conceded_last5': team_row['team_avg_conceded_last5'],
        'opp_avg_goals_last5': opp_row['team_avg_goals_last5'],
        'opp_avg_conceded_last5': opp_row['team_avg_conceded_last5'],
        'goals_diff_last5': team_row['team_avg_goals_last5'] - opp_row['team_avg_goals_last5'],
        'conceded_diff_last5': team_row['team_avg_conceded_last5'] - opp_row['team_avg_conceded_last5'],
        'win_rate_diff': team_row['team_win_rate_last5'] - opp_row['team_win_rate_last5']
    }])

    win_prob = model.predict_proba(match_features)[0][1]
    not_win_prob = 1 - win_prob

    return {
        "Match": f"{team} vs {opponent}",
        f"{team} Win Probability": round(win_prob, 3),
        f"{team} Not Win Probability": round(not_win_prob, 3)
    }

In [8]:
latest_team_stats = df.sort_values(by='date').groupby('team').last().reset_index()
latest_team_stats = latest_team_stats[['team', 'team_avg_goals_last5', 'team_avg_conceded_last5', 'team_win_rate_last5']]

predict_match("Brazil", "France")

{'Match': 'Brazil vs France',
 'Brazil Win Probability': np.float64(0.099),
 'Brazil Not Win Probability': np.float64(0.901)}

In [10]:
predict_match("Argentina", "England")


{'Match': 'Argentina vs England',
 'Argentina Win Probability': np.float64(0.182),
 'Argentina Not Win Probability': np.float64(0.818)}

In [11]:
predict_match("United States", "Mexico")

{'Match': 'United States vs Mexico',
 'United States Win Probability': np.float64(0.325),
 'United States Not Win Probability': np.float64(0.675)}

In [12]:
predict_match("Spain", "Germany")

{'Match': 'Spain vs Germany',
 'Spain Win Probability': np.float64(0.358),
 'Spain Not Win Probability': np.float64(0.642)}

In [13]:
matches_to_predict = [
    ("Brazil", "France"),
    ("Argentina", "England"),
    ("Spain", "Germany"),
    ("United States", "Mexico"),
    ("Portugal", "Netherlands")
]

predictions = []

for team, opponent in matches_to_predict:
    result = predict_match(team, opponent)
    predictions.append(result)

predictions_df = pd.DataFrame(predictions)
predictions_df

,Match,Brazil Win Probability,Brazil Not Win Probability,Argentina Win Probability,Argentina Not Win Probability,Spain Win Probability,Spain Not Win Probability,United States Win Probability,United States Not Win Probability,Portugal Win Probability,Portugal Not Win Probability
0,Brazil vs France,0.099,0.901,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Argentina vs England,NaN,NaN,0.182,0.818,NaN,NaN,NaN,NaN,NaN,NaN
2,Spain vs Germany,NaN,NaN,NaN,NaN,0.358,0.642,NaN,NaN,NaN,NaN
3,United States vs Mexico,NaN,NaN,NaN,NaN,NaN,NaN,0.325,0.675,NaN,NaN
4,Portugal vs Netherlands,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.084,0.916


In [14]:
predictions_df.to_csv("world_cup_match_predictions.csv", index=False)
files.download("world_cup_match_predictions.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>